In [1]:
import os
import random
import shutil
from PIL import Image
from sklearn.model_selection import train_test_split

print("✅ All imports successful!")

✅ All imports successful!


In [2]:
# Source folders (already processed in Phase 1 and 1b)
COLOR_PATH     = r"D:\Development\8th Sem Project\TomatoClassification\dataset\processed"
SEGMENTED_PATH = r"D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented"

# Output folder for mixed dataset
OUTPUT_PATH    = r"D:\Development\8th Sem Project\TomatoClassification\dataset\processed_mixed"

IMAGE_SIZE           = (224, 224)
IMAGES_PER_SOURCE    = 500   # 500 color + 500 segmented = 1000 per class
SEED                 = 42

print(f"✅ Paths configured!")
print(f"📁 Color source     : {COLOR_PATH}")
print(f"📁 Segmented source : {SEGMENTED_PATH}")
print(f"📁 Mixed output     : {OUTPUT_PATH}")
print(f"🔢 Per source       : {IMAGES_PER_SOURCE} images")
print(f"🔢 Total per class  : {IMAGES_PER_SOURCE * 2} images")

✅ Paths configured!
📁 Color source     : D:\Development\8th Sem Project\TomatoClassification\dataset\processed
📁 Segmented source : D:\Development\8th Sem Project\TomatoClassification\dataset\processed_segmented
📁 Mixed output     : D:\Development\8th Sem Project\TomatoClassification\dataset\processed_mixed
🔢 Per source       : 500 images
🔢 Total per class  : 1000 images


In [3]:
# Get classes from both folders
color_classes     = sorted([
    f for f in os.listdir(os.path.join(COLOR_PATH, 'train'))
    if os.path.isdir(os.path.join(COLOR_PATH, 'train', f))
])
segmented_classes = sorted([
    f for f in os.listdir(os.path.join(SEGMENTED_PATH, 'train'))
    if os.path.isdir(os.path.join(SEGMENTED_PATH, 'train', f))
])

# Verify both folders have same classes
if color_classes == segmented_classes:
    TOMATO_CLASSES = color_classes
    print(f"✅ Classes match perfectly! Found {len(TOMATO_CLASSES)} classes\n")
    for cls in TOMATO_CLASSES:
        print(f"  → {cls.replace('Tomato___', '')}")
else:
    print("❌ ERROR — Classes don't match!")
    print(f"Color only    : {set(color_classes) - set(segmented_classes)}")
    print(f"Segmented only: {set(segmented_classes) - set(color_classes)}")

✅ Classes match perfectly! Found 10 classes

  → Bacterial_spot
  → Early_blight
  → Late_blight
  → Leaf_Mold
  → Septoria_leaf_spot
  → Spider_mites Two-spotted_spider_mite
  → Target_Spot
  → Tomato_Yellow_Leaf_Curl_Virus
  → Tomato_mosaic_virus
  → healthy


In [4]:
print("📊 Available Images Per Class\n" + "="*60)
print(f"{'Class':<40} {'Color':>8} {'Segmented':>12} {'Total':>8}")
print("-"*60)

for cls in TOMATO_CLASSES:
    # Count all images across train+val+test
    color_count = sum(
        len(os.listdir(os.path.join(COLOR_PATH, split, cls)))
        for split in ['train', 'val', 'test']
        if os.path.exists(os.path.join(COLOR_PATH, split, cls))
    )
    seg_count = sum(
        len(os.listdir(os.path.join(SEGMENTED_PATH, split, cls)))
        for split in ['train', 'val', 'test']
        if os.path.exists(os.path.join(SEGMENTED_PATH, split, cls))
    )
    name = cls.replace("Tomato___", "")
    print(f"{name:<40} {color_count:>8} {seg_count:>12} {color_count+seg_count:>8}")

print("="*60)
print(f"\n✅ Will take {IMAGES_PER_SOURCE} from each source per class")
print(f"✅ Final mixed dataset: {IMAGES_PER_SOURCE * 2} images per class")

📊 Available Images Per Class
Class                                       Color    Segmented    Total
------------------------------------------------------------
Bacterial_spot                               1000         1000     2000
Early_blight                                 1000         1000     2000
Late_blight                                  1000         1000     2000
Leaf_Mold                                     952          952     1904
Septoria_leaf_spot                           1000         1000     2000
Spider_mites Two-spotted_spider_mite         1000         1000     2000
Target_Spot                                  1000         1000     2000
Tomato_Yellow_Leaf_Curl_Virus                1000         1000     2000
Tomato_mosaic_virus                           373          373      746
healthy                                      1000         1000     2000

✅ Will take 500 from each source per class
✅ Final mixed dataset: 1000 images per class


In [5]:
print("🔄 Creating Mixed Dataset...\n")

split_counts = {'train': {}, 'val': {}, 'test': {}}

for cls in TOMATO_CLASSES:
    
    # --- Collect images from COLOR (all splits combined) ---
    color_images = []
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(COLOR_PATH, split, cls)
        if os.path.exists(split_dir):
            for img in os.listdir(split_dir):
                if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                    color_images.append(os.path.join(split_dir, img))

    # --- Collect images from SEGMENTED (all splits combined) ---
    seg_images = []
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(SEGMENTED_PATH, split, cls)
        if os.path.exists(split_dir):
            for img in os.listdir(split_dir):
                if img.lower().endswith(('.jpg', '.jpeg', '.png')):
                    seg_images.append(os.path.join(split_dir, img))

    # --- Sample 500 from each source ---
    random.seed(SEED)
    color_sample = random.sample(color_images, 
                                  min(IMAGES_PER_SOURCE, len(color_images)))
    seg_sample   = random.sample(seg_images,   
                                  min(IMAGES_PER_SOURCE, len(seg_images)))

    # --- Combine both ---
    all_images = color_sample + seg_sample
    random.shuffle(all_images)

    # --- Split 70/15/15 ---
    train_imgs, temp_imgs = train_test_split(all_images, 
                                              test_size=0.30, 
                                              random_state=SEED)
    val_imgs, test_imgs   = train_test_split(temp_imgs,  
                                              test_size=0.50, 
                                              random_state=SEED)

    splits = {'train': train_imgs, 'val': val_imgs, 'test': test_imgs}

    # --- Copy & resize to output ---
    for split_name, img_list in splits.items():
        dest_dir = os.path.join(OUTPUT_PATH, split_name, cls)
        os.makedirs(dest_dir, exist_ok=True)
        split_counts[split_name][cls] = len(img_list)

        for idx, src_path in enumerate(img_list):
            # Rename to avoid color/segmented filename conflicts
            ext = os.path.splitext(src_path)[1]
            dst = os.path.join(dest_dir, f"mixed_{idx:04d}{ext}")

            img = Image.open(src_path).convert("RGB")
            img = img.resize(IMAGE_SIZE, Image.LANCZOS)
            img.save(dst)

print("✅ Mixed dataset created successfully!")

🔄 Creating Mixed Dataset...

✅ Mixed dataset created successfully!


In [6]:
print("📁 Mixed Dataset Structure:\n")
print(f"{'Class':<45} {'Train':>8} {'Val':>8} {'Test':>8}")
print("-" * 72)

for cls in TOMATO_CLASSES:
    name = cls.replace("Tomato___", "")
    t  = split_counts['train'].get(cls, 0)
    v  = split_counts['val'].get(cls, 0)
    te = split_counts['test'].get(cls, 0)
    print(f"{name:<45} {t:>8} {v:>8} {te:>8}")

print("-" * 72)
total_train = sum(split_counts['train'].values())
total_val   = sum(split_counts['val'].values())
total_test  = sum(split_counts['test'].values())
print(f"{'TOTAL':<45} {total_train:>8} {total_val:>8} {total_test:>8}")
print(f"\n📊 Grand Total : {total_train + total_val + total_test} images")
print(f"📂 Saved to    : {OUTPUT_PATH}")
print(f"\n✅ Mixed Preprocessing Complete!")

📁 Mixed Dataset Structure:

Class                                            Train      Val     Test
------------------------------------------------------------------------
Bacterial_spot                                     700      150      150
Early_blight                                       700      150      150
Late_blight                                        700      150      150
Leaf_Mold                                          700      150      150
Septoria_leaf_spot                                 700      150      150
Spider_mites Two-spotted_spider_mite               700      150      150
Target_Spot                                        700      150      150
Tomato_Yellow_Leaf_Curl_Virus                      700      150      150
Tomato_mosaic_virus                                522      112      112
healthy                                            700      150      150
------------------------------------------------------------------------
TOTAL                  